# Feature Engineering & Preprocessing

**Task 1 · Adey Innovations Fraud Detection**

This notebook:
1. Loads the cleaned datasets from EDA
2. Engineers features for Fraud_Data
3. Encodes and scales both datasets
4. Handles class imbalance (SMOTE)
5. Saves model-ready splits to `data/processed/`


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from feature_engineering import engineer_fraud_features
from preprocessing       import (
    encode_and_scale_fraud, scale_creditcard,
    stratified_split, apply_smote, apply_combined_resampling,
    FRAUD_TARGET, CC_TARGET,
)
from eda_utils import summarise_imbalance

print('Imports OK')


## 1. Load Cleaned Data

> Run `eda-fraud-data.ipynb` and `eda-creditcard.ipynb` first to generate the cleaned CSVs.

In [ ]:
PROC_DIR = '../data/processed'

df_fraud = pd.read_csv(f'{PROC_DIR}/fraud_data_clean.csv', parse_dates=['signup_time','purchase_time'])
df_cc    = pd.read_csv(f'{PROC_DIR}/creditcard_clean.csv')

print("Fraud_Data shape:", df_fraud.shape)
print("CreditCard shape:", df_cc.shape)


## 2. Feature Engineering — Fraud_Data

In [ ]:
# Apply all feature engineering
df_fraud_fe = engineer_fraud_features(df_fraud)

new_features = ['hour_of_day','day_of_week','time_since_signup_s','time_since_signup_h',
                'user_txn_count','user_mean_purchase','device_txn_count']
print("New features added:")
print(df_fraud_fe[new_features].describe().round(3))


In [ ]:
# Visualise key engineered features vs fraud rate
import matplotlib.ticker as mtick

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# time_since_signup
for ax_i, col, xlabel in [
    (0, 'time_since_signup_h', 'Hours since signup (capped 720h)'),
    (1, 'user_txn_count',      'Transactions per user'),
    (2, 'device_txn_count',    'Transactions per device'),
]:
    for cls, label, color in [(0,'Legit','#2196F3'),(1,'Fraud','#E53935')]:
        vals = df_fraud_fe[df_fraud_fe[FRAUD_TARGET]==cls][col]
        if col == 'time_since_signup_h':
            vals = vals[vals <= 720]
        vals.plot.kde(ax=axes[ax_i], label=label, color=color, linewidth=2)
    axes[ax_i].set_title(col)
    axes[ax_i].set_xlabel(xlabel)
    axes[ax_i].legend()

plt.suptitle('Engineered Features: Fraud vs Legitimate', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 3. Encode & Scale — Fraud_Data

In [ ]:
df_fraud_enc, scaler_fraud, feature_cols_fraud = encode_and_scale_fraud(
    df_fraud_fe, scaler_type='standard'
)

print(f"Encoded shape: {df_fraud_enc.shape}")
print(f"Features ({len(feature_cols_fraud)}):", feature_cols_fraud[:10], "...")


## 4. Scale — CreditCard

V1–V28 are already PCA-scaled. We only scale `Time` and `Amount`.

In [ ]:
df_cc_enc, scaler_cc = scale_creditcard(df_cc, scaler_type='standard')

print("Amount stats after scaling:")
print(df_cc_enc['Amount'].describe().round(3))


## 5. Train / Test Split (Stratified)

In [ ]:
# --- Fraud_Data ---
X_train_f, X_test_f, y_train_f, y_test_f = stratified_split(
    df_fraud_enc, target_col=FRAUD_TARGET, test_size=0.2, random_state=42
)

# --- CreditCard ---
X_train_c, X_test_c, y_train_c, y_test_c = stratified_split(
    df_cc_enc, target_col=CC_TARGET, test_size=0.2, random_state=42
)

print("\n== Fraud_Data ==")
print(f"  Train: {len(X_train_f):,}  |  Test: {len(X_test_f):,}")
print(f"  Train fraud: {y_train_f.sum():,} ({y_train_f.mean()*100:.1f}%)")

print("\n== CreditCard ==")
print(f"  Train: {len(X_train_c):,}  |  Test: {len(X_test_c):,}")
print(f"  Train fraud: {y_train_c.sum():,} ({y_train_c.mean()*100:.3f}%)")


## 6. Handle Class Imbalance — SMOTE

**Justification:** We use SMOTE (Synthetic Minority Over-sampling Technique) because:
- Pure undersampling throws away majority-class data, losing valuable information.
- SMOTE generates synthetic minority samples via k-NN interpolation, preserving the distribution of the feature space.
- It is applied **only to the training set** — the test set is kept as-is to simulate real-world evaluation.
- For CreditCard (0.17% fraud), we use a combined SMOTE + undersampling pipeline to avoid an excessively large training set.

In [ ]:
# ── Fraud_Data: SMOTE only (fraud ~9.4%, moderate imbalance)
X_res_f, y_res_f = apply_smote(X_train_f, y_train_f, sampling_strategy=0.4, random_state=42)
summarise_imbalance(y_train_f, y_res_f, label='(Fraud_Data)')


In [ ]:
# ── CreditCard: combined SMOTE + undersampling (extreme imbalance)
X_res_c, y_res_c = apply_combined_resampling(
    X_train_c, y_train_c,
    over_strategy=0.1,   # SMOTE up to 10% minority
    under_strategy=0.5,  # undersample majority to 2× minority
    random_state=42,
)
summarise_imbalance(y_train_c, y_res_c, label='(CreditCard)')


In [ ]:
# Visualise resampled distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (y_b, y_a, title) in zip(axes, [
    (y_train_f, y_res_f, 'Fraud_Data'),
    (y_train_c, y_res_c, 'CreditCard'),
]):
    x = ['Before', 'After']
    b_fraud = [y_b.mean()*100, y_a.mean()*100]
    ax.bar(x, b_fraud, color=['#2196F3','#E53935'], alpha=0.85, edgecolor='white')
    ax.set_title(f'{title} — Fraud % Before vs After Resampling')
    ax.set_ylabel('Fraud Rate (%)')
    for i, v in enumerate(b_fraud):
        ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()


## 7. Save Model-Ready Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# Fraud_Data
pd.concat([X_train_f, y_train_f], axis=1).to_csv(f'{PROC_DIR}/fraud_train_raw.csv', index=False)
pd.concat([X_test_f,  y_test_f],  axis=1).to_csv(f'{PROC_DIR}/fraud_test.csv',      index=False)

X_res_f_df = pd.DataFrame(X_res_f, columns=X_train_f.columns)
X_res_f_df[FRAUD_TARGET] = y_res_f.values
X_res_f_df.to_csv(f'{PROC_DIR}/fraud_train_smote.csv', index=False)

# CreditCard
pd.concat([X_train_c, y_train_c], axis=1).to_csv(f'{PROC_DIR}/cc_train_raw.csv', index=False)
pd.concat([X_test_c,  y_test_c],  axis=1).to_csv(f'{PROC_DIR}/cc_test.csv',      index=False)

X_res_c_df = pd.DataFrame(X_res_c, columns=X_train_c.columns)
X_res_c_df[CC_TARGET] = y_res_c.values
X_res_c_df.to_csv(f'{PROC_DIR}/cc_train_resampled.csv', index=False)

# Save scalers
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler_fraud, '../models/scaler_fraud.pkl')
joblib.dump(scaler_cc,    '../models/scaler_cc.pkl')

print("All processed files saved to data/processed/")
print("Scalers saved to models/")


## 8. Summary of Task 1 Decisions

| Step | Decision | Justification |
|---|---|---|
| Missing values | No imputation needed | Both datasets are complete |
| Duplicates | Drop exact duplicates | Prevents data leakage and overfitting |
| Geolocation | merge_asof range lookup | Efficient O(n log n) approach for IP range matching |
| Feature: time_since_signup | Seconds → Hours | Hours are more interpretable; fraud spikes within first hour |
| Feature: velocity | Per-user & per-device counts | High velocity is a known fraud signal |
| Encoding | One-hot (no ordinal assumption) | source, browser, sex, country have no natural order |
| Scaling | StandardScaler | Centres features; better for Logistic Regression baseline |
| Imbalance (Fraud_Data) | SMOTE, strategy=0.4 | Moderate 9.4% fraud rate; generates synthetic samples |
| Imbalance (CreditCard) | SMOTE + Undersampling | Extreme 0.17% rate; pure SMOTE creates oversized dataset |
| Test set | Never resampled | Preserves real-world distribution for honest evaluation |
